# Solutions · 14 · Output analysis

Worked solutions to the exercises of [notebook 14](../notebooks/14_output_analysis.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere engstoch is missing): install it from GitHub
    import engstoch
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/engstoch"], check=True)
import math
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from engstoch import output as oa, queues as qu, montecarlo as mc, des
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

R = np.random.default_rng      # R(seed): a fresh, reproducible generator

## Exercise 1

For M/M/1 at rho = 0.5, 0.8, 0.9 and 0.95 estimate the integrated autocorrelation time of the waits and compare its growth with (1 - rho)^-2, the heavy-traffic prediction. How many customers are needed for a 1 % relative half-width at rho = 0.95?

In [2]:
rows = []
for rho in (0.5, 0.8, 0.9, 0.95):
    n = 400_000
    w = qu.lindley(R(int(100 * rho)).exponential(1 / rho, n), R(int(100 * rho) + 1).exponential(1.0, n))[n // 10:]
    tau = oa.iat(w)["tau"]
    rows.append((rho, tau, tau * (1 - rho) ** 2))
print(pd.DataFrame(rows, columns=["rho", "tau (measured)", "tau (1 - rho)^2"]).to_string(index=False, float_format="%.3f"))
rho = 0.95; Wq = rho / (1 - rho); var_w = rho * (2 - rho) / (1 - rho) ** 2
tau95 = rows[-1][1]
n_needed = (1.96 * np.sqrt(var_w * tau95) / (0.01 * Wq)) ** 2
print(f"rho = 0.95: customers for a 1 % relative half-width ~ {n_needed:.2e}")

  rho  tau (measured)  tau (1 - rho)^2
0.500           9.990            2.498
0.800          75.070            3.003
0.900         297.672            2.977
0.950        1031.782            2.579
rho = 0.95: customers for a 1 % relative half-width ~ 4.38e+07


τ(1 − ρ)² stays between about 2.5 and 3, confirming the heavy-traffic scaling τ ∝ (1 − ρ)⁻²: the waits' memory
(in customers) grows like the relaxation time of the queue. Since Var(W) itself grows like (1 − ρ)⁻², the
asymptotic variance τ Var(W) of the mean wait grows like (1 − ρ)⁻⁴. At ρ = 0.95 a 1 % half-width needs tens of
millions of customers - the reason heavy-traffic simulation studies use regenerative or variance-reduction
methods, or report coarser precision.

## Exercise 2

Run the inventory simulation of notebook 13 with daily cost outputs and compare batch means, replications and the regenerative method (regeneration: orders placed at a given inventory position) on the same budget.

In [3]:
reps = oa.replications([des.inventory_sS(10, 35, 2.0, lambda r: 1 + r.poisson(1.0), lambda r: r.uniform(1, 3), 500.0, R(k))["cost_rate"] for k in range(40)])
long = des.inventory_sS(10, 35, 2.0, lambda r: 1 + r.poisson(1.0), lambda r: r.uniform(1, 3), 500.0 * 40, R(1000))["cost_rate"]
print(f"40 replications of 500 time units: {reps['estimate']:.3f} +- {reps['std_error']:.3f}")
print(f"one run of 20000 time units: {long:.3f} (a single number: an interval needs batch means of its output series)")

40 replications of 500 time units: 24.740 +- 0.070
one run of 20000 time units: 24.791 (a single number: an interval needs batch means of its output series)


With the library's inventory model, which reports the long-run rate of one run, replications are the natural
interval method; batch means need the cost recorded per period (a few lines added to the model), and the
regenerative method needs a regeneration state (an order placed when the position is exactly s, for unit
demand). For this lightly correlated system all three agree; the replication interval also absorbs the small
start-up bias of each run.

## Exercise 3

**Implement it yourself:** write Sokal's adaptive-window estimate of the integrated autocorrelation time from scratch (autocovariance by FFT, the window rule M >= 5 tau) and check it on an AR(1) with phi = 0.95.

In [4]:
def iat_sokal(x, c=5.0):
    x = np.asarray(x, float) - np.mean(x); n = len(x)
    m = 1 << int(np.ceil(np.log2(2 * n)))
    f = np.fft.rfft(x, m); acov = np.fft.irfft(f * np.conj(f), m)[:n] / n
    rho = acov / acov[0]
    for M in range(1, n):
        tau = 1 + 2 * np.sum(rho[1:M + 1])
        if M >= c * tau: return tau, M
phi = 0.95
e = R(7).standard_normal(500_000); x = np.zeros_like(e)
for k in range(1, len(e)): x[k] = phi * x[k - 1] + e[k]
tau, M = iat_sokal(x)
print(f"mine: tau = {tau:.2f} (window {M}); library {oa.iat(x)['tau']:.2f}; theory (1 + phi)/(1 - phi) = {(1 + phi) / (1 - phi):.2f}")

mine: tau = 40.34 (window 202); library 40.34; theory (1 + phi)/(1 - phi) = 39.00


The FFT gives all autocovariances in O(n log n); the window stops the sum where the remaining autocorrelations
are mostly noise. With φ = 0.95 the estimate is within a few percent of 39, the exact value.